# SimTrigger: イベントを集める条件

`EdepPickUp` の記録から、DAQ が集めるイベントかどうかを決めます。条件を満たさないイベントでは `AS_SKIP` を返すので、
**後ろのモジュール(TPCEmulator, HAEmulator, RawWriter2p)はそのイベントを処理しません**。設定ファイルの `response` の最初のモジュールです。

| | |
| --- | --- |
| SMILE_tool_box | `SMILE_2p` の `DAQ_Mode`, `Collect_Number` |
| ソース | `simulation_source/modules/SimTrigger/` |

## 1. 何をするか

TPC のステップの有無(`tpc`)、シンチのステップの有無(`psa`)、アンチのエネルギーが 0 より大きいか(`anti`)で決めます。

| `daq_mode` | 集める条件 | 実データでは |
| --- | --- | --- |
| 0 | 全部 | |
| 1 | `tpc` | TPC モード |
| 2 | `psa` | |
| 3 | `tpc && psa` | **ETCC モード**(TPC とシンチの同時計数) |
| 4 | `tpc && psa && !anti` | ETCC モード + アンチの反同時計数 |
| 5 | `tpc && !psa` | |
| 6 | `psa && !tpc` | |

`collect_number` が 0 より大きいと、その数だけ集めた次のイベントで終了します(`-n` より先に止まる)。

閾値はここではかけません(ステップがあれば「あり」)。電荷の閾値は TPCEmulator、ADC の閾値は HAEmulator が後でかけます。

## 2. パラメータ

| パラメータ | 型 | 既定値(C++) | この設定の値 | 単位 | 説明 |
| --- | --- | --- | --- | --- | --- |
| `daq_mode` | int | 3 | 3 |  | 集めるイベント(1 TPC、2 シンチ、3 TPC && シンチ、4 + anti なし、5 TPC だけ、6 シンチだけ、0 全部) |
| `collect_number` | int | 0 | 0 |  | この数だけ集めたら終了(0: 制限なし) |
パラメータは設定ファイル `config/simulation/smile2p_flight_parma_thermal_neutron_d10_raw.json` の `response[SimTrigger].parameters` に書きます
(書き方は `config/README.md`)。1 回だけ変えて試すときは `run_sim.py ... --set response.SimTrigger.daq_mode=1` で上書きします。
「既定値(C++)」は設定ファイルに書かなかったときの値です。

## 3. ほかのモジュールとのやり取り

受け取るもの: `EdepPickUp` の `TPCStepE().size()`, `ScintiStepE().size()`, `AntiEnergy()`。

| 関数 | 型 | 中身 | 使うモジュール |
| --- | --- | --- | --- |
| `Collected()` | long | ここまでに集めたイベントの数 | (終わりに表示) |

後ろのモジュールに値は渡さず、`AS_SKIP` でイベントを止めるだけです。

## 4. 確かめる

`edep` から、`daq_mode` ごとに何イベント集まるかを数えます(実行したのは `daq_mode = 3`)。

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_sim.py").exists())
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CONFIG = "smile2p_flight_parma_thermal_neutron_d10_raw"
cfg = ec.load_sim(CONFIG)
# 1_DataIO/DataIO_example.ipynb の出力(d10 の熱中性子、100 万個、seed 1)を使う
SIM = REPO / "tutorials_simulation" / "cache" / "d10_seed1_1M"
EDEP, RAW = SIM / "edep.root", SIM / "raw"
assert EDEP.exists(), "先に ../../DataIO_example.ipynb を実行してください"

def flat(column):
    # RDataFrame.AsNumpy() の vector 型の列 -> 全イベントをつないだ 1 次元の numpy 配列
    return np.concatenate([np.asarray(v) for v in column]) if len(column) else np.array([])

In [2]:
d = ROOT.RDataFrame("edep", str(EDEP)).Define("ntpc", "tpc_e.size()").Define("nsci", "sci_e.size()").AsNumpy(["ntpc", "nsci", "e_anti"])
tpc, psa, anti = d["ntpc"] > 0, d["nsci"] > 0, d["e_anti"] > 0
modes = {1: tpc, 2: psa, 3: tpc & psa, 4: tpc & psa & ~anti, 5: tpc & ~psa, 6: psa & ~tpc}
for m, sel in modes.items():
    print(f"daq_mode {m}: {sel.sum():6d} events")
print("(daq_mode 0 は全 1000000 個。edep には TPC かシンチにステップのあるイベントだけが入っている)")
print("".join(l for l in open(SIM / "run_sim.log") if l.startswith("SimTrigger:")))

daq_mode 1:    310 events
daq_mode 2:   9153 events
daq_mode 3:    270 events
daq_mode 4:    210 events
daq_mode 5:     40 events
daq_mode 6:   8883 events
(daq_mode 0 は全 1000000 個。edep には TPC かシンチにステップのあるイベントだけが入っている)
SimTrigger: daq_mode 3, collected 270 of 1000000 events

